In [ ]:
texto = "Hello, world! I can't believe it."
tokens_simples = texto.split()
print(tokens_simples)
# ['Hello,', 'world!', 'I', "can't", 'believe', 'it.']

['Hello,', 'world!', 'I', "can't", 'believe', 'it.']


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize, sent_tokenize

texto = "Dr. Smith arrived at 3 p.m. yesterday. He bought coffee. What a surprise!"

# Sentence segmentation
oraciones = sent_tokenize(texto)
print(oraciones)
# ['Dr. Smith arrived at 3 p.m. yesterday.', 'He bought coffee.', 'What a surprise!']

# Word tokenization
print("Tokens")
tokens = word_tokenize(oraciones[2])
print(tokens)
# ['Dr.', 'Smith', 'arrived', 'at', '3', 'p.m.', 'yesterday', '.']

['Dr. Smith arrived at 3 p.m. yesterday.', 'He bought coffee.', 'What a surprise!']
Tokens
['What', 'a', 'surprise', '!']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp(texto)

print([sent.text for sent in doc.sents])
print([token.text for token in doc])
print("Morfema")
print([token.morph for token in doc])
print("Lema")
print([token.lemma for token in doc])
print([token.lemma_ for token in doc])
print([token.pos for token in doc])
print([token.pos_ for token in doc])
print([spacy.explain(token.pos_) for token in doc])






['Dr. Smith arrived at 3 p.m. yesterday.', 'He bought coffee.', 'What a surprise!']
['Dr.', 'Smith', 'arrived', 'at', '3', 'p.m.', 'yesterday', '.', 'He', 'bought', 'coffee', '.', 'What', 'a', 'surprise', '!']
Morfema
[Number=Sing, Number=Sing, Tense=Past|VerbForm=Fin, , NumType=Card, Number=Sing, Number=Sing, PunctType=Peri, Case=Nom|Gender=Masc|Number=Sing|Person=3|PronType=Prs, Tense=Past|VerbForm=Fin, Number=Sing, PunctType=Peri, , Definite=Ind|PronType=Art, Number=Sing, PunctType=Peri]
Lema
[12921814047719899021, 769169189799025663, 15357328220926964755, 11667289587015813222, 602994839685422785, 367645567677572359, 1756787072497230782, 12646065887601541794, 1655312771067108281, 9457496526477982497, 3197928453018144401, 12646065887601541794, 5865838185239622912, 11901859001352538922, 12150403604591081639, 17494803046312582752]
['Dr.', 'Smith', 'arrive', 'at', '3', 'p.m.', 'yesterday', '.', 'he', 'buy', 'coffee', '.', 'what', 'a', 'surprise', '!']
[<univ_pos_t.PROPN: 96>, <univ_pos_

In [ ]:
!python -m spacy download es_core_news_sm
import spacy

nlp = spacy.load("es_core_news_sm")

def pos_puro_espanol_sin_universales(token):
    """
    Construye las categorías gramaticales desde cero,
    sin tocar jamás las 17 categorías universales (token.pos_).
    """
    # Extraemos el diccionario morfológico puro (ej: {'Gender': 'Fem', 'Number': 'Plur'})
    morph = token.morph.to_dict()

    # 1. IDENTIFICACIÓN PURA DE VERBOS (Por rasgos de forma o modo verbal)
    if "VerbForm" in morph:
        form = morph["VerbForm"]
        if form == "Part": return "PARTICIPIO_NATIVO"
        if form == "Ger":  return "GERUNDIO_NATIVO"
        if form == "Inf":  return "INFINITIVO_NATIVO"

    if "Mood" in morph:
        mood = morph["Mood"]
        if mood == "Sub":  return "VERBO_MODO_SUBJUNTIVO"
        if mood == "Imp":  return "VERBO_MODO_IMPERATIVO"
        if mood == "Ind":  return "VERBO_MODO_INDICATIVO"

    # 2. IDENTIFICACIÓN PURA DE PRONOMBRES Y DETERMINANTES (Por el tipo de pronombre)
    if "PronType" in morph:
        ptype = morph["PronType"]
        if ptype == "Art":
            # Separamos artículos por definición real, no por etiqueta universal
            return "ARTÍCULO_DETERMINADO" if morph.get("Definite") == "Def" else "ARTÍCULO_INDETERMINADO"
        if ptype == "Prs": return "PRONOMBRE_PERSONAL_NATIVO"
        if ptype == "Rel": return "PRONOMBRE_RELATIVO_NATIVO"
        if ptype == "Dem": return "DEMOSTRATIVO_NATIVO"
        if ptype == "Ind": return "INDEFINIDO_NATIVO"
        if ptype == "Int": return "INTERROGATIVO_NATIVO"

    # 3. SUSTANTIVOS Y ADJETIVOS (Se identifican por su combinación de género/número/tipo)
    # En el modelo AnCora de spaCy, el rasgo 'NounType' define la sustancia real
    if "NounType" in morph:
        return "SUSTANTIVO_PROPIO_NATIVO" if morph["NounType"] == "Prop" else "SUSTANTIVO_COMÚN_NATIVO"

    # Si tiene género y número pero no es verbo ni pronombre, es un adjetivo o sustantivo básico
    if "Gender" in morph and "Number" in morph:
        # Si el token se comporta morfológicamente como modificador, lo marcamos
        return "ADJETIVO_CALIFICATIVO_NATIVO"

    # 4. ELEMENTOS ESTRUCTURALES O INVARIABLES (Si no tienen morfología flexible)
    # Caemos en un análisis de texto crudo para signos o espacios
    if token.text.isspace(): return "ESPACIO"
    if token.text in [".", ",", ";", ":", "!", "¡", "?", "¿"]: return "PUNTUACIÓN_NATIVA"
    if token.text.isdigit(): return "NÚMERO_DIGITAL"

    # Para palabras invariables sin rasgos de flexión (como preposiciones o adverbios cortos)
    # recurrimos a la única alternativa fuera de spaCy: un chequeo rápido de longitud/contexto
    # o devolvemos una categoría base limpia.
    return "PARTÍCULA_INVARIABLE_NATIVA"

# =====================================================================
# EJECUCIÓN CON TU LIST COMPREHENSION REAL
# =====================================================================
texto = "El gato que corrió estaba cantando. ¡Ojalá tú lo veas mañana!"
doc = nlp(texto)

# Tu list comprehension llamando al motor morfológico puro
mis_etiquetas_propias = [pos_puro_espanol_sin_universales(token) for token in doc]

print(f"{'Palabra':<12} | {'Tu Sistema POS 100% Independiente'}")
print("-" * 55)
for token, mi_pos in zip(doc, mis_etiquetas_propias):
    print(f"{token.text:<12} | {mi_pos}")


  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Palabra      | Tu Sistema POS 100% Independiente
-------------------------------------------------------
El           | ARTÍCULO_DETERMINADO
gato         | ADJETIVO_CALIFICATIVO_NATIVO
que          | PARTÍCULA_INVARIABLE_NATIVA
corrió       | VERBO_MODO_INDICATIVO
estaba       | VERBO_MODO_INDICATIVO
cantando     | GERUNDIO_NATIVO
.            | PUNTUACIÓN_NATIVA
¡            | PUNTUACIÓN_NATIVA
Ojalá        | PARTÍCULA_INVARIABLE_NATIVA
tú           | PRONOMBRE_PERSONAL_NATIVO
lo           | PRONOMBRE_PERSONAL_

In [ ]:
import spacy

# Al cargar el modelo, ya estás cargando el corpus AnCora codificado
nlp = spacy.load("es_core_news_sm")

texto = "El coche amado de María avanzaba mientras ella cantando un tres % de canciones ¡Ojalá tú lo veas mañana!"
doc = nlp(texto)

# TU LIST COMPREHENSION: Extrae directamente la morfología nativa del corpus
# sin pasar por las 17 etiquetas universales del POS
corpus_espanol = [token.morph.to_dict() for token in doc]

# Mostramos el resultado para que veas que la base de datos morfológica ya viene incluida
print(f"{'Palabra':<12} | Base de datos morfológica integrada (AnCora)")
print("-" * 75)
for token, caracteristicas in zip(doc, corpus_espanol):
    print(f"{token.text:<12} | {caracteristicas}")

Palabra      | Base de datos morfológica integrada (AnCora)
---------------------------------------------------------------------------
El           | {'Definite': 'Def', 'Gender': 'Masc', 'Number': 'Sing', 'PronType': 'Art'}
coche        | {'Gender': 'Masc', 'Number': 'Sing'}
amado        | {'Gender': 'Masc', 'Number': 'Sing', 'VerbForm': 'Part'}
de           | {}
María        | {}
avanzaba     | {'Mood': 'Ind', 'Number': 'Sing', 'Person': '3', 'Tense': 'Imp', 'VerbForm': 'Fin'}
mientras     | {}
ella         | {'Case': 'Acc,Nom', 'Gender': 'Fem', 'Number': 'Sing', 'Person': '3', 'PronType': 'Prs'}
cantando     | {'VerbForm': 'Ger'}
un           | {'Definite': 'Ind', 'Gender': 'Masc', 'Number': 'Sing', 'PronType': 'Art'}
tres         | {'NumType': 'Card'}
%            | {'NumForm': 'Digit'}
de           | {}
canciones    | {'Gender': 'Fem', 'Number': 'Plur'}
¡            | {'PunctSide': 'Ini', 'PunctType': 'Excl'}
Ojalá        | {}
tú           | {'Case': 'Nom', 'Number': 'Sing', 'Per

In [ ]:
import spacy

# Asegúrate de haber ejecutado en tu terminal: python -m spacy download es_core_news_sm
nlp = spacy.load("es_core_news_sm")

# Frase de prueba con alta complejidad morfológica en español
texto = "El coche amado de María avanzaba mientras ella cantando un tres % de canciones ¡Ojalá tú lo veas mañana!"
doc = nlp(texto)

# TU LIST COMPREHENSION DIRECTA USANDO ANCORA
# Extrae el tag detallado y pide la explicación oficial del corpus
pos_ancora = [spacy.explain(token.tag_) for token in doc]

# Mostramos el resultado en una tabla limpia
print(f"{'Palabra':<12} | {'Tag AnCora':<10} | Categoría POS Detallada (AnCora)")
print("-" * 80)
for token, tag, descripcion in zip(doc, [t.tag_ for t in doc], pos_ancora):
    print(f"{token.text:<12} | {tag:<10} | {descripcion}")


Palabra      | Tag AnCora | Categoría POS Detallada (AnCora)
--------------------------------------------------------------------------------
El           | DET        | determiner
coche        | NOUN       | noun
amado        | ADJ        | adjective
de           | ADP        | adposition
María        | PROPN      | proper noun
avanzaba     | VERB       | verb
mientras     | SCONJ      | subordinating conjunction
ella         | PRON       | pronoun
cantando     | VERB       | verb
un           | DET        | determiner
tres         | NUM        | numeral
%            | SYM        | symbol
de           | ADP        | adposition
canciones    | NOUN       | noun
¡            | PUNCT      | punctuation
Ojalá        | INTJ       | interjection
tú           | PRON       | pronoun
lo           | PRON       | pronoun
veas         | VERB       | verb
mañana       | ADV        | adverb
!            | PUNCT      | punctuation


In [ ]:
import spacy

# Recuerda instalarlo antes en tu terminal: python -m spacy download es_core_news_sm
nlp = spacy.load("es_core_news_sm")

# Tablas del estándar AnCora para traducir cada posición del tag_
TRADUCCION_ANCORA = {
    "categorias": {"v": "Verbo", "n": "Sustantivo", "a": "Adjetivo", "p": "Pronombre", "d": "Determinante", "r": "Adverbio", "s": "Preposición", "c": "Conjunción"},
    "tipo_verbo": {"m": "Principal", "a": "Auxiliar", "s": "Semiauxiliar"},
    "modo":       {"i": "Indicativo", "s": "Subjuntivo", "m": "Imperativo", "n": "Infinitivo", "g": "Gerundio", "p": "Participio"},
    "tiempo":     {"p": "Presente", "i": "Imperfecto", "f": "Futuro", "s": "Pasado", "c": "Condicional"},
    "persona":    {"1": "1ª pers.", "2": "2ª pers.", "3": "3ª pers."},
    "genero":     {"m": "Masculino", "f": "Femenino", "c": "Común"},
    "numero":     {"s": "Singular", "p": "Plural", "n": "Invariable"},
    "tipo_sust":  {"c": "Común", "p": "Propio"},
    "tipo_adj":   {"q": "Calificativo", "o": "Ordinal"},
    "caso":       {"n": "Nominativo", "a": "Acusativo", "d": "Dativo", "o": "Oblicuo"}
}

def mi_pos_ancora(token):
    tag = token.tag_  # Ej: "vmii3s0" o "ncfs000"

    if not tag or len(tag) < 3:
        return "Otro"

    inicial = tag[0]
    cat = TRADUCCION_ANCORA["categorias"].get(inicial, "Otro")

    # 1. Mapeo completo para Verbos
    if inicial == "v" and len(tag) >= 6:
        tipo = TRADUCCION_ANCORA["tipo_verbo"].get(tag[1], "")
        modo = TRADUCCION_ANCORA["modo"].get(tag[2], "")
        if tag[2] in ["i", "s", "m"]:  # Si es conjugado
            tiempo = TRADUCCION_ANCORA["tiempo"].get(tag[3], "")
            pers = TRADUCCION_ANCORA["persona"].get(tag[4], "")
            num = TRADUCCION_ANCORA["numero"].get(tag[5], "")
            return f"{cat} {tipo} ({modo} {tiempo}, {pers} {num})"
        return f"{cat} {tipo} ({modo})"

    # 2. Mapeo completo para Sustantivos
    elif inicial == "n" and len(tag) >= 5:
        tipo = TRADUCCION_ANCORA["tipo_sust"].get(tag[1], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[2], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[3], "")
        return f"{cat} {tipo} ({gen} {num})"

    # 3. Mapeo completo para Adjetivos
    elif inicial == "a" and len(tag) >= 5:
        tipo = TRADUCCION_ANCORA["tipo_adj"].get(tag[1], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[3], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[4], "")
        return f"{cat} {tipo} ({gen} {num})"

    # 4. Mapeo completo para Pronombres Personales
    elif tag.startswith("pp") and len(tag) >= 6:
        pers = TRADUCCION_ANCORA["persona"].get(tag[2], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[3], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[4], "")
        caso = TRADUCCION_ANCORA["caso"].get(tag[5], "No especificado")
        return f"Pronombre Personal ({pers}, {gen} {num}, Caso: {caso})"

    return cat

# --- PRUEBA CON TU FRASE ---
texto = "El coche amado avanzaba mientras ella lo miraba."
doc = nlp(texto)

# TU LIST COMPREHENSION DIRECTA AHORA SÍ MOSTRARÁ TODO EL DETALLE REAL
pos_espanol_detallado = [mi_pos_ancora(token) for token in doc]

# Imprimimos el resultado en pantalla
for token, pos in zip(doc, pos_espanol_detallado):
    print(f"{token.text:<12} -> {pos}")


El           -> Otro
coche        -> Otro
amado        -> Otro
avanzaba     -> Otro
mientras     -> Otro
ella         -> Otro
lo           -> Otro
miraba       -> Otro
.            -> Otro


In [ ]:
import re
import unicodedata
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

# Descargas obligatorias de NLTK (solo se hacen la primera vez)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 1. TEXTO DE PRUEBA (Tu corpus del laboratorio)
texto_sucio = "¡Hola Mundo! El PLN es fantástico... . Él jugó. ¿Estás listo para aprender en 2026? Áéíóú."

# 2. NORMALIZACIÓN UNICODE (Remover acentos y estandarizar caracteres)
texto_normalizado = "".join(
    c for c in unicodedata.normalize('NFD', texto_sucio)
    if unicodedata.category(c) != 'Mn'
).lower() # Convertimos a minúsculas también

# 3. EXPRESIONES REGULARES (Eliminar números y signos de puntuación)
# Dejamos solo letras y espacios en blanco
texto_limpio = re.sub(r'[^a-z\s]', '', texto_normalizado)
print("Diferentes textos")
print(texto_sucio)
print(texto_normalizado)
print(texto_limpio)
# 4. TOKENIZACIÓN (Segmentación de palabras)
tokens = word_tokenize(texto_limpio)

# 5. REMOCIÓN DE STOPWORDS (Palabras vacías en español)
palabras_vacias = set(stopwords.words('spanish'))
tokens_finales = [token for token in tokens if token not in palabras_vacias]

# --- RESULTADOS DEL LABORATORIO ---
print("--- PIPELINE DE PLN TRADICIONAL ---")
print(f"Texto Original:    {texto_sucio}")
print(f"Texto Limpio:      {texto_limpio}")
print(f"Tokens extraídos:  {tokens}")
print(f"Tokens sin stopwords: {tokens_finales}")


Diferentes textos
¡Hola Mundo! El PLN es fantástico... . Él jugó. ¿Estás listo para aprender en 2026? Áéíóú.
¡hola mundo! el pln es fantastico... . el jugo. ¿estas listo para aprender en 2026? aeiou.
hola mundo el pln es fantastico  el jugo estas listo para aprender en  aeiou
--- PIPELINE DE PLN TRADICIONAL ---
Texto Original:    ¡Hola Mundo! El PLN es fantástico... . Él jugó. ¿Estás listo para aprender en 2026? Áéíóú.
Texto Limpio:      hola mundo el pln es fantastico  el jugo estas listo para aprender en  aeiou
Tokens extraídos:  ['hola', 'mundo', 'el', 'pln', 'es', 'fantastico', 'el', 'jugo', 'estas', 'listo', 'para', 'aprender', 'en', 'aeiou']
Tokens sin stopwords: ['hola', 'mundo', 'pln', 'fantastico', 'jugo', 'listo', 'aprender', 'aeiou']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
!pip install minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import io
import re
import unicodedata
from minio import Minio
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Descargas obligatorias del diccionario de NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 1. Configuracion de MinIO (Endpoint limpio sin rutas al final)
client = Minio(
    endpoint=    "159.195.245.124:9000",          # IP pública de tu VPS o servidor local
    access_key="admin_vps_user",    # Usuario asignado
    secret_key="123456780",  # Contraseña asignada
    secure=False               # Ponlo en True si usas HTTPS (SSL)

)

bucket_name = "archivoeq1"
object_name = "BDSINIESTROSReducida-anonymized.xlsx"
response = None

# 2. Extraccion segura de los datos para evitar fallos de inicializacion
try:
      # Obtener la lista de todos los buckets
    buckets = client.list_buckets()

    print("--- LISTA DE BUCKETS DISPONIBLES ---")
    for bucket in buckets:
        print(f"Nombre: {bucket.name} | Creado el: {bucket.creation_date}")
    response = client.get_object(bucket_name, object_name)
    datos_binarios = response.read()

    # Cargar el binario directamente en Pandas sin guardarlo en el disco
    excel_memoria = io.BytesIO(datos_binarios)
    df = pd.read_excel(excel_memoria)

    # Convertimos cada columna a string de forma independiente, ignorando filas vacías (NaN)
    lista_de_palabras = []
    for col in df.columns:
        # Convertimos la columna a texto y quitamos los valores nulos o "nan"
        textos_columna = df[col].astype(str).dropna().tolist()
        lista_de_palabras.extend(textos_columna)

    # Unimos todo en una sola cadena de texto gigante
    texto_crudo = " ".join(lista_de_palabras)

    print("Texto extraído con éxito desde el archivo Excel de MinIO.")
    print(f"Tamaño de la cadena obtenida: {len(texto_crudo)} caracteres.\n")

    print("Texto extraído con éxito desde el archivo Excel de MinIO.\n")
    response = client.get_object(bucket_name, object_name)
    #texto_crudo = response.read().decode('utf-8')
    print("Archivo descargado con exito desde MinIO.\n")
except Exception as e:
    print("Error al conectar o descargar de MinIO:", e)
    texto_crudo = ""
finally:
    if response is not None:
        response.close()
        response.release_conn()

# 3. Pipeline de PLN con NLTK (Solo corre si el texto se descargo correctamente)
if texto_crudo:
    # 1. Normalizacion Unicode y conversion a minusculas
    texto_normalizado = "".join(
        c for c in unicodedata.normalize('NFD', texto_crudo)
        if unicodedata.category(c) != 'Mn'
    ).lower()

    # 2. Expresiones regulares para limpiar signos de puntuacion y numeros
    texto_limpio = re.sub(r'[^a-z\s]', '', texto_normalizado)

    # 3. Tokenizacion de palabras mediante NLTK
    tokens = word_tokenize(texto_limpio)

    # 4. Remocion de Stopwords en espanol
    palabras_vacias = set(stopwords.words('spanish'))

    # Filtramos las stopwords y tambien el residuo 'nan' propio de Pandas
    tokens_finales = [
        t for t in tokens
        if t not in palabras_vacias and t != 'nan' and len(t) > 1
    ]

    print("--- TOKENS FINALES DEL EXCEL ---")
    print(tokens_finales)
else:
    print("No se pudo ejecutar el pipeline de PLN porque el texto extraido esta vacio.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


--- LISTA DE BUCKETS DISPONIBLES ---
Nombre: 23432423424 | Creado el: 2026-09-08 08:35:41.883000+00:00
Nombre: andresolano | Creado el: 2026-09-19 14:22:14.127000+00:00
Nombre: archivoeq1 | Creado el: 2026-09-12 15:54:30.697000+00:00
Nombre: ayastaefrain | Creado el: 2026-09-19 13:50:56.421000+00:00
Nombre: cachiquejairo | Creado el: 2026-09-19 13:52:24.075000+00:00
Nombre: carlosayala | Creado el: 2026-09-19 14:00:52.576000+00:00
Nombre: carlosmartinez | Creado el: 2026-09-19 13:55:36.031000+00:00
Nombre: chiroqueomar | Creado el: 2026-09-19 13:59:59.567000+00:00
Nombre: laboratorio-narrativas | Creado el: 2026-09-14 06:09:34.188000+00:00
Nombre: lazaroronald | Creado el: 2026-09-19 13:50:42.709000+00:00
Nombre: mavelramos | Creado el: 2026-09-19 14:20:24.837000+00:00
Nombre: merajhoanna | Creado el: 2026-09-19 13:59:07.802000+00:00
Nombre: olivermalpartida | Creado el: 2026-09-19 13:59:35.461000+00:00
Nombre: quispemanuel | Creado el: 2026-09-19 13:50:16.805000+00:00
Nombre: rafaelin

In [2]:
!pip install minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 99.3 MB/s eta 0:00:00


In [5]:
!pip install feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 8.1 MB/s eta 0:00:00


In [8]:
import re
import duckdb
import requests
from bs4 import BeautifulSoup
import feedparser
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# --- CONFIGURACIÓN Y DESCARGAS DE NLTK ---
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('tokenizers/punkt_tab')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
    nltk.download('stopwords')

# --- CONFIGURACIÓN DE MEDIOS ---
# Cambiado a tipo "html" ya que se usan las portadas web principales
MEDIOS = {
    "moscow_times": {
        "url": "https://themoscowtimes.com",
        "tipo": "html"
    },
    "rt": {
        "url": "https://rt.com",
        "tipo": "html"
    }
}

# --- FUNCIÓN PARA LIMPIAR CARACTERES INVÁLIDOS ---
def limpiar_caracteres_invalidos(texto_bytes):
    """Remueve caracteres de control que rompen los lectores XML tradicionales."""
    texto = texto_bytes.decode('utf-8', errors='ignore')
    invalid_xml_chars = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]')
    return invalid_xml_chars.sub('', texto)

# --- FUNCIÓN DE PROCESAMIENTO DE TEXTO ---
def procesar_texto(texto):
    """Limpia el texto, remueve stopwords y extrae tokens básicos."""
    if not texto:
        return ""
    tokens = word_tokenize(texto.lower())
    stop_words = set(stopwords.words('english'))
    tokens_filtrados = [w for w in tokens if w.isalnum() and w not in stop_words]
    return " ".join(tokens_filtrados)

# --- SCRAPER PRINCIPAL ---
def ejecutar_scraping():
    datos_extraidos = []
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    for medio, info in MEDIOS.items():
        print(f"Iniciando scraping de {medio}...")
        try:
            response = requests.get(info["url"], headers=headers, timeout=15)
            response.raise_for_status()

            contenido_limpio = limpiar_caracteres_invalidos(response.content)

            if info["tipo"] == "rss":
                feed = feedparser.parse(contenido_limpio)

                if feed.entries:
                    for entrada in feed.entries:
                        titulo = entrada.get("title", "").strip()
                        enlace = entrada.get("link", "").strip()
                        resumen = entrada.get("summary", "") or entrada.get("description", "")

                        if titulo:
                            texto_limpio = procesar_texto(f"{titulo} {resumen}")
                            datos_extraidos.append((medio, titulo, enlace, texto_limpio))

                if not feed.entries or len(datos_extraidos) == 0:
                    soup = BeautifulSoup(contenido_limpio, 'lxml-xml')
                    items = soup.find_all(['item', 'entry'])

                    for item in items:
                        titulo_tag = item.find(['title', 'media:title'])
                        enlace_tag = item.find(['link', 'guid'])
                        desc_tag = item.find(['description', 'summary', 'content'])

                        titulo = titulo_tag.get_text().strip() if titulo_tag else ""
                        enlace = ""
                        if enlace_tag:
                            enlace = enlace_tag.get('href') or enlace_tag.get_text()
                        enlace = enlace.strip() if enlace else info["url"]
                        desc = desc_tag.get_text().strip() if desc_tag else ""

                        if titulo:
                            texto_limpio = procesar_texto(f"{titulo} {desc}")
                            datos_extraidos.append((medio, titulo, enlace, texto_limpio))

            else:
                # Procesamiento robusto para páginas HTML usando BeautifulSoup y lxml
                soup = BeautifulSoup(contenido_limpio, 'lxml')

                if medio == "moscow_times":
                    # Extrae titulares desde las etiquetas h3 del portal
                    titulos_h3 = soup.find_all('h3')
                    for h3 in titulos_h3:
                        titulo = h3.get_text(strip=True)
                        # Busca si el elemento superior o hijo contiene el enlace
                        enlace_tag = h3.find('a') or h3.find_parent('a')
                        enlace = enlace_tag['href'] if enlace_tag and enlace_tag.has_attr('href') else info["url"]

                        # Asegura URLs absolutas
                        if enlace.startswith('/'):
                            enlace = info["url"] + enlace

                        if titulo and len(titulo) > 10:  # Evita capturar textos de menú cortos
                            texto_limpio = procesar_texto(titulo)
                            datos_extraidos.append((medio, titulo, enlace, texto_limpio))

                elif medio == "rt":
                    # Extrae links y textos de los bloques principales de noticias
                    links = soup.find_all('a', href=re.compile(r'/russia/|/world-news/|/business/|/sport/|/pop-culture/'))
                    for link in links:
                        titulo = link.get_text(strip=True)
                        enlace = link['href']

                        if enlace.startswith('/'):
                            enlace = info["url"] + enlace

                        if titulo and len(titulo) > 15:
                            texto_limpio = procesar_texto(titulo)
                            datos_extraidos.append((medio, titulo, enlace, texto_limpio))

            print(f"Procesamiento exitoso de {medio}.")

        except Exception as e:
            print(f"Fallo en el procesamiento de {medio}: {e}")

    # --- CARGA EN DUCKDB ---
    if datos_extraidos:
        print(f"\nAlimentando DuckDB con {len(datos_extraidos)} registros...")
        con = duckdb.connect('noticias.db')

        con.execute("""
            CREATE TABLE IF NOT EXISTS noticias (
                fuente VARCHAR,
                titulo VARCHAR,
                url VARCHAR,
                texto_procesado VARCHAR
            )
        """)

        con.executemany("""
            INSERT INTO noticias VALUES (?, ?, ?, ?)
        """, datos_extraidos)

        total = con.execute("SELECT COUNT(*) FROM noticias").fetchone()
        print(f"Datos guardados con éxito. Total de filas en DuckDB: {total[0]}")
        con.close()
    else:
        print("\nNo se generaron datos para alimentar DuckDB.")

if __name__ == "__main__":
    ejecutar_scraping()


Iniciando scraping de moscow_times...
Procesamiento exitoso de moscow_times.
Iniciando scraping de rt...
Procesamiento exitoso de rt.

Alimentando DuckDB con 65 registros...
Datos guardados con éxito. Total de filas en DuckDB: 65


In [11]:
import duckdb

con = duckdb.connect('noticias.db')
resultados = con.execute("SELECT fuente, titulo, texto_procesado FROM noticias LIMIT 100").fetchall()

print("Primeros 5 registros en la base de datos:")
for fila in resultados:
    print(f"Fuente: {fila[0]}")
    print(f"Titulo: {fila[1]}")
    print(f"Texto procesado (NLTK): {fila[2]}")
    print("-" * 40)

con.close()

Primeros 5 registros en la base de datos:
Fuente: moscow_times
Titulo: Russia Holds State Duma, Regional Elections
Texto procesado (NLTK): russia holds state duma regional elections
----------------------------------------
Fuente: moscow_times
Titulo: 3 Dead in Moscow Region, Drones Hit Oil Refinery in Russian Capital
Texto procesado (NLTK): 3 dead moscow region drones hit oil refinery russian capital
----------------------------------------
Fuente: moscow_times
Titulo: ‘Writing Was Reclaiming That Power’: Lana Estemirova on Her Mother and the Chechen Wars
Texto procesado (NLTK): writing reclaiming power lana estemirova mother chechen wars
----------------------------------------
Fuente: moscow_times
Titulo: Russia Has a Labor Shortage. So Why Can’t Anyone Find a Job?
Texto procesado (NLTK): russia labor shortage anyone find job
----------------------------------------
Fuente: moscow_times
Titulo: In Photos: Russians Vote in the First Wartime Parliamentary Elections
Texto procesado (NL

In [14]:
import math
import re
import duckdb
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter

# --- DESCARGAS COMPLEMENTARIAS DE NLTK ---
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

def analizar_fuentes():
    # Conectar a la base de datos local
    con = duckdb.connect('noticias.db')

    # Verificar si la tabla tiene datos
    existe_tabla = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name = 'noticias'").fetchone()[0]
    if not existe_tabla or con.execute("SELECT COUNT(*) FROM noticias").fetchone()[0] == 0:
        print("La tabla 'noticias' está vacía o no existe en noticias.db. Ejecuta el scraper primero.")
        con.close()
        return

    # Leer las filas agrupadas por fuente
    filas = con.execute("SELECT fuente, titulo FROM noticias").fetchall()
    con.close()

    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # Estructuras para almacenar tokens por fuente
    documentos_por_fuente = {"moscow_times": [], "rt": []}

    # Procesamiento de texto: Extracción de Tokens y Morfemas (Lematización)
    for fuente, titulo in filas:
        if fuente not in documentos_por_fuente:
            continue

        # Limpieza básica y tokenización
        tokens_crudos = word_tokenize(titulo.lower())

        # Filtrado de palabras vacías y caracteres no alfanuméricos
        tokens_limpios = [t for t in tokens_crudos if t.isalnum() and t not in stop_words]

        # Reducción morfológica básica (Lematización)
        morfemas = [lemmatizer.lemmatize(t) for t in tokens_limpios]

        # Agrupamos por fuente en forma de "documento" completo
        documentos_por_fuente[fuente].extend(morfemas)

    # --- CÁLCULO DE TF (Frecuencia de Término Relativa) ---
    tf_por_fuente = {}
    vocabulario_total = set()

    for fuente, tokens in documentos_por_fuente.items():
        total_tokens = len(tokens)
        conteo = Counter(tokens)
        tf_por_fuente[fuente] = {palabra: cant / total_tokens for palabra, cant in conteo.items()}
        vocabulario_total.update(conteo.keys())

    # --- CÁLCULO DE IDF (Frecuencia Inversa de Documento) ---
    # En este contexto, tratamos a cada fuente como un "gran documento" corporativo (N = 2)
    N = len(documentos_por_fuente)
    idf = {}

    for palabra in vocabulario_total:
        # Contamos en cuántas fuentes aparece la palabra
        apariciones_en_fuentes = sum(1 for fuente in documentos_por_fuente if palabra in documentos_por_fuente[fuente])
        # Fórmula clásica logarítmica (ajustada para evitar división por cero si no aparece)
        idf[palabra] = math.log(N / (apariciones_en_fuentes if apariciones_en_fuentes > 0 else 1)) + 1

    # --- RESULTADOS Y AGRUPACIÓN ---
    print("MÉTRICAS POR FUENTE (TOP 10 PALABRAS CLAVE POR TF-IDF)\n")

    for fuente in documentos_por_fuente.keys():
        print(f"=== FUENTE: {fuente.upper()} ===")
        print(f"Total tokens/morfemas procesados: {len(documentos_por_fuente[fuente])}")

        # Calcular TF-IDF final combinando los diccionarios anteriores
        tf_idf_fuente = {}
        for palabra in tf_por_fuente[fuente]:
            tf_idf_fuente[palabra] = tf_por_fuente[fuente][palabra] * idf[palabra]

        # Ordenar por el valor TF-IDF más alto
        top_palabras = sorted(tf_idf_fuente.items(), key=lambda x: x[1], reverse=True)[:30]

        print(f"{'Palabra (Morfema)':<20} | {'TF (Frec.)':<12} | {'IDF':<10} | {'TF-IDF':<10}")
        print("-" * 60)
        for palabra, valor_tfidf in top_palabras:
            valor_tf = tf_por_fuente[fuente][palabra]
            valor_idf = idf[palabra]
            print(f"{palabra:<20} | {valor_tf:<12.5f} | {valor_idf:<10.5f} | {valor_tfidf:<10.5f}")
        print("\n")

if __name__ == "__main__":
    analizar_fuentes()


MÉTRICAS POR FUENTE (TOP 10 PALABRAS CLAVE POR TF-IDF)

=== FUENTE: MOSCOW_TIMES ===
Total tokens/morfemas procesados: 432
Palabra (Morfema)    | TF (Frec.)   | IDF        | TF-IDF    
------------------------------------------------------------
russia               | 0.03935      | 1.00000    | 0.03935   
russian              | 0.03241      | 1.00000    | 0.03241   
life                 | 0.01620      | 1.69315    | 0.02744   
region               | 0.01157      | 1.69315    | 0.01960   
photo                | 0.00926      | 1.69315    | 0.01568   
east                 | 0.00926      | 1.69315    | 0.01568   
political            | 0.00926      | 1.69315    | 0.01568   
shortage             | 0.00694      | 1.69315    | 0.01176   
vote                 | 0.00694      | 1.69315    | 0.01176   
kremlin              | 0.00694      | 1.69315    | 0.01176   
calling              | 0.00694      | 1.69315    | 0.01176   
prison               | 0.00694      | 1.69315    | 0.01176   
survival  

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
